In [ ]:
import pandas as pd
import numpy as np
import janitor
from lets_plot import *
LetsPlot.setup_html()

In [ ]:
raw_1854 = pd.read_excel("raw_night_1854_1855.xlsx")
raw_1856 = pd.read_excel("raw_night_1855_1856.xlsx")

In [ ]:
raw_1854.head()

# Explanation

This code converts the dataset from wide to long format.

- `pivot_longer()` combines the three death-cause columns into two columns.
- `names_to = "disease"` stores the cause of death category.
- `values_to = "count"` stores the monthly death counts.

# Why this is useful

Many visualization tools work better with long format data.

What this code is trying to achieve

To prepare the dataset for visualization.

In [ ]:
raw2 = raw_1854["Month"].str.split(" ", expand=True).rename(columns={0: "month_2", 1: "year"})
raw  = pd.concat([raw_1854, raw2], axis=1)

raw3 = raw.pivot_longer(
    index=["Month", "Avg Army Size", "month_2", "year"],
    names_to="disease",
    values_to="count"
)

raw3.head()

In [ ]:
raw3["sqrt_count"] = np.sqrt(raw3["count"])

raw3.head()

## Step 1: Order categories

This ensures the months appear in chronological order starting from July,
and the disease categories stack in a consistent order on each wedge.

In [ ]:
raw3["month_2"] = pd.Categorical(
    raw3["month_2"],
    categories=["Jul","Aug","Sep","Oct","Nov","Dec","Jan","Feb","Mar","Apr","May","Jun"],
    ordered=True
)

raw3["disease"] = pd.Categorical(
    raw3["disease"],
    categories=["Wounds", "Other Causes", "Disease Deaths"],
    ordered=True
)

## Step 2: Create a bar chart

- `month_2` = month on the x-axis
- `sqrt_count` = square-root scaled death count (so smaller categories stay visible)
- `fill` = colour groups for each cause of death
- `geom_bar(stat='identity')` uses the actual values from the dataset to draw bars
- `scale_fill_manual(values=["#D4A0A0", "#6B6B6B", "#8FBCD4"])` — pink for wounds, grey for other causes, blue for disease
- `theme(axis_text_y = element_blank())` removes y-axis labels to simplify the visualization

In [ ]:
(
    ggplot(raw3, aes("month_2", "sqrt_count", fill="disease"))
    + geom_bar(stat='identity', alpha=0.6)
    + scale_fill_manual(
        values=["#D4A0A0", "#6B6B6B", "#8FBCD4"],
        labels=["Wounds", "Other Causes", "Disease Deaths"]
    )
    + theme(axis_text_y=element_blank())
)

## Circular Chart (Rose Diagram)

`coord_polar()` wraps the bar chart into a polar coordinate system, creating the rose diagram Florence Nightingale used to argue that disease — not battle wounds — was the primary killer of soldiers.

In [ ]:
p1 = (
    ggplot(raw3, aes("month_2", "sqrt_count", fill="disease"))
    + geom_bar(stat='identity', alpha=0.6, show_legend=True)
    + scale_fill_manual(
        values=["#D4A0A0", "#6B6B6B", "#8FBCD4"],
        labels=["Wounds", "Other Causes", "Disease Deaths"]
    )
    + coord_polar()
    + labs(
        fill="Cause of Death",
        title="Diagram of Causes of Mortality in the Army",
        subtitle="APRIL 1854 TO MARCH 1855",
        caption="Inspired by Florence Nightingale (1858)"
    )
    + theme(
        axis_title_y=element_blank(),
        axis_title_x=element_blank(),
        axis_text_y=element_blank(),
        axis_text_x=element_text(size=10, color="black", face="bold"),
        axis_ticks=element_blank(),
        plot_subtitle=element_text(hjust=0.5, size=10, color="grey40"),
        plot_title=element_text(hjust=0.5, face="bold", size=13),
        plot_caption=element_text(hjust=0.5, color="grey50", size=8),
        panel_grid=element_blank(),
        panel_border=element_blank(),
        legend_position="right",
        legend_title=element_text(face="bold"),
        plot_background=element_rect(fill="#F5EFE0"),
        panel_background=element_rect(fill="#F5EFE0")
    )
    + ggsize(900, 400)
)
p1

## Second period: April 1855 to March 1856

Same pipeline applied to the second dataset, covering the period after sanitary reforms were introduced.

In [ ]:
raw5 = raw_1856["Month"].str.split(" ", expand=True).rename(columns={0: "month_2", 1: "year"})
raw4 = raw_1856.pivot_longer(
    index=["Month", "Avg Army Size"],
    names_to="disease",
    values_to="count"
)
raw6 = pd.concat([raw4, raw5], axis=1)
raw6["sqrt_count"] = np.sqrt(raw6["count"])

raw6["month_2"] = pd.Categorical(
    raw6["month_2"],
    categories=["Jul","Aug","Sep","Oct","Nov","Dec","Jan","Feb","Mar","Apr","May","Jun"],
    ordered=True
)
raw6["disease"] = pd.Categorical(
    raw6["disease"],
    categories=["Wounds", "Other Causes", "Disease Deaths"],
    ordered=True
)

raw6.head()

In [ ]:
p2 = (
    ggplot(raw6, aes("month_2", "sqrt_count", fill="disease"))
    + geom_bar(stat='identity', alpha=0.6, show_legend=True)
    + scale_fill_manual(
        values=["#D4A0A0", "#6B6B6B", "#8FBCD4"],
        labels=["Wounds", "Other Causes", "Disease Deaths"]
    )
    + coord_polar()
    + labs(
        fill="Cause of Death",
        title="Diagram of Causes of Mortality in the Army",
        subtitle="APRIL 1855 TO MARCH 1856",
        caption="Inspired by Florence Nightingale (1858)"
    )
    + theme(
        axis_title_y=element_blank(),
        axis_title_x=element_blank(),
        axis_text_y=element_blank(),
        axis_text_x=element_text(size=10, color="black", face="bold"),
        axis_ticks=element_blank(),
        plot_subtitle=element_text(hjust=0.5, size=10, color="grey40"),
        plot_title=element_text(hjust=0.5, face="bold", size=13),
        plot_caption=element_text(hjust=0.5, color="grey50", size=8),
        panel_grid=element_blank(),
        panel_border=element_blank(),
        legend_position="right",
        legend_title=element_text(face="bold"),
        plot_background=element_rect(fill="#F5EFE0"),
        panel_background=element_rect(fill="#F5EFE0")
    )
    + ggsize(900, 400)
)
p2

In [ ]:
gggrid([p1, p2], ncol=2)